# Phase 2C: counterfactual data collection (Kaggle CPU)

This notebook collects **E9 paired simulator branches carrying E1 (zone damper) and E2 (reheat valve) zone-subset excitation**. It implements the approved design in `configs/phase2c/collection.json` and `docs/phase2c_data_collection_proposal.md`:
- 30 scenarios, **300 branch points × 20 arms**, 3 h horizon;
- 30 % random zone subsets with 15/30/60-min holds;
- bit-exact branching with global-RNG restore and common random numbers;
- effective-action and no-op tracking.

It reuses the Phase 2B branching harness (`src/counterfactual/*`).

## How to run (two sessions)

**Kaggle settings:** Accelerator = **None (CPU)**, Internet = **On**. Add the input dataset containing `phase1_sample.zip` (the same one used for Phase 2B training).

**Session 1**
1. Run all cells (*Save Version → Save & Run All (Commit)*).
2. Collection stops starting new branch points after `TIME_BUDGET_H` and keeps everything finished.

**Session 2** (only if QA says *INCOMPLETE*)
1. *Add Input → Notebooks →* this notebook's Session 1 version. Its `/kaggle/working/phase2c_counterfactual` then appears under `/kaggle/input/...`.
2. Run all again. Cell 4 copies the finished branch points; the collector skips them and finishes the rest. No duplicates are created: one file per branch point, written atomically, and completeness is checked before skipping.

**Output** (`/kaggle/working`):
- `phase2c_counterfactual/` with `branches/*.npz`, `plan.json`, `manifest.jsonl`, `qa_report.json`, `session_logs/`, `logs/`;
- `phase2c_counterfactual.zip`.

Download the zip and unzip it at the repo root as `data/phase2c_counterfactual/`.

In [ ]:
# ---- parameters -------------------------------------------------------------
import os, sys, glob, json, shutil, subprocess, time
BRANCH = "phase2c"                       # git branch with the Phase 2C code
STARTER_KIT_REV = "1fb75606d12e016463fbc63de704967c5b2bf9d5"   # revision the Phase 1 data was collected with
PARALLEL = 4                             # Kaggle CPU sessions have 4 cores; one worker per scenario
TIME_BUDGET_H = 10.5                     # stop STARTING branch points after this (12 h session limit)
DRY_RUN = False                          # True: 1 scenario x 2 branch points into OUT_DRY (~2 min), for a smoke test

os.environ["MPLBACKEND"] = "Agg"       # the kernel's inline backend is not importable in the /tmp venv subprocesses
LOCAL_TEST = os.environ.get("P2C_LOCAL_TEST")          # set only by the local validation harness
if LOCAL_TEST:                           # local harness: reuse the checked-out repo and its venv
    REPO = os.environ["P2C_REPO"]; PY = os.environ["P2C_PY"]; WORK_OUT = os.environ["P2C_OUT"]; INPUT_ROOT = os.environ["P2C_INPUT"]
else:
    REPO = "/tmp/p2c/smart-building-rl"; PY = "/tmp/p2c/venv/bin/python"; WORK_OUT = "/kaggle/working"; INPUT_ROOT = "/kaggle/input"
OUT = os.path.join(WORK_OUT, "phase2c_counterfactual_dryrun" if DRY_RUN else "phase2c_counterfactual")
CONFIG = f"{REPO}/configs/phase2c/collection.json"
os.makedirs(OUT, exist_ok=True); os.makedirs(f"{OUT}/logs", exist_ok=True)
print("REPO", REPO, "| OUT", OUT, "| DRY_RUN", DRY_RUN)


In [ ]:
# ---- 1. environment (code + pinned simulator) ---------------------------------
if not LOCAL_TEST:
    !mkdir -p /tmp/p2c && rm -rf {REPO} && git clone -q -b {BRANCH} https://github.com/fannam/smart-building-rl.git {REPO}
    !pip install -q uv
    !uv venv -q --python 3.12 /tmp/p2c/venv
    !git clone -q https://github.com/smart-buildings-2026/starter-kit.git {REPO}/starter-kit && cd {REPO}/starter-kit && git checkout -q {STARTER_KIT_REV}
    !uv pip install -q -p {PY} -e {REPO}/starter-kit
    !uv pip install -q -p {PY} "numpy==2.0.2" "pandas==2.3.3"
!cd {REPO} && git log --oneline -1
!{PY} -c "import numpy, scipy, numba, smart_buildings; print('numpy', numpy.__version__, 'scipy', scipy.__version__, 'numba', numba.__version__)"


In [ ]:
# ---- 2. Phase 1 baseline episodes (branch-point planning + bit-exact replay reference) ----
dst = f"{REPO}/data/phase1_sample"
if not os.path.exists(f"{dst}/manifest.jsonl"):
    os.makedirs(f"{REPO}/data", exist_ok=True)
    zips = glob.glob(f"{INPUT_ROOT}/**/phase1_sample.zip", recursive=True)
    dirs = [d for d in glob.glob(f"{INPUT_ROOT}/**/phase1_sample", recursive=True) if os.path.isfile(f"{d}/manifest.jsonl")]
    if dirs:
        os.symlink(dirs[0], dst); print("linked", dirs[0])
    elif zips:
        subprocess.run(["unzip", "-q", zips[0], "-d", f"{REPO}/data"], check=True); print("unzipped", zips[0])
    else:
        print("phase1_sample not attached -> regenerating the 30 baseline episodes deterministically (~30-40 min)")
        subprocess.run(f"cd {REPO} && MPLBACKEND=Agg {PY} -u src/collect_data.py --out data/phase1_sample --profiles baseline "
                       f"--episodes 1 --steps 2016 --seed 0 --workers {PARALLEL}", shell=True, check=True)
rows = [json.loads(l) for l in open(f"{dst}/manifest.jsonl")]
n_base = sum(r["profile"] == "baseline" for r in rows)
assert n_base == 30, f"expected 30 baseline episodes, found {n_base}"
print("baseline episodes:", n_base)


In [ ]:
# ---- 3. tests (fast; includes a real simulator reproducibility test, ~1 min). STOPS the notebook on failure ----
t = subprocess.run(f"cd {REPO} && {PY} -m unittest tests.test_phase2c", shell=True, capture_output=True, text=True)
print("\n".join((t.stdout + t.stderr).splitlines()[-6:]))
if t.returncode != 0:
    print((t.stdout + t.stderr)[-6000:])
    raise RuntimeError("tests failed -> not collecting (see output above)")


In [ ]:
# ---- 4. resume: import branch points finished in a previous session ----------
prev = [d for d in glob.glob(f"{INPUT_ROOT}/**/{os.path.basename(OUT)}/branches", recursive=True)
        if os.path.realpath(d) != os.path.realpath(f"{OUT}/branches")]
os.makedirs(f"{OUT}/branches", exist_ok=True)
copied = 0
for d in prev:
    for f in glob.glob(f"{d}/*.npz"):
        t = f"{OUT}/branches/{os.path.basename(f)}"
        if not os.path.exists(t):
            shutil.copy2(f, t + ".part"); os.replace(t + ".part", t); copied += 1
    for sub in ("session_logs", "logs"):
        for f in glob.glob(f"{os.path.dirname(d)}/{sub}/*"):
            os.makedirs(f"{OUT}/{sub}", exist_ok=True)
            if not os.path.exists(f"{OUT}/{sub}/{os.path.basename(f)}"):
                shutil.copy2(f, f"{OUT}/{sub}/")
print(f"previous-session sources: {prev or 'none'}; imported {copied} branch files; now {len(glob.glob(f'{OUT}/branches/*.npz'))} present")


In [ ]:
# ---- 5. validate configuration + plan (no simulation) ----------------------
SCEN = "--scenarios building_train_8/atlanta_1 --n-points 2" if DRY_RUN else "--scenarios all"
!cd {REPO} && {PY} scripts/phase2c_collect.py --config {CONFIG} --out {OUT} {SCEN} --parallel {PARALLEL} --validate-only


In [ ]:
# ---- 6. collect (resumable; stops starting new branch points after TIME_BUDGET_H) ----
log = f"{OUT}/logs/collect_{int(time.time())}.log"
print("log:", log)
!cd {REPO} && {PY} -u scripts/phase2c_collect.py --config {CONFIG} --out {OUT} {SCEN} --parallel {PARALLEL} --time-budget-hours {TIME_BUDGET_H} 2>&1 | grep -v -i warning | tee {log} | grep -E "== |SESSION|FAIL|Traceback|Error"
sess = sorted(glob.glob(f"{OUT}/session_logs/session_*.json"))
if sess:
    res = json.load(open(sess[-1]))["results"]
    fatal = [r for r in res if "fatal" in r]
    if fatal:
        print(f"{len(fatal)} scenario(s) failed; first error:\n" + fatal[0]["fatal"][-3000:])
    if res and len(fatal) == len(res):
        raise RuntimeError("every scenario failed -> fix the error above before resuming")


In [ ]:
# ---- 7. post-collection QA + manifest ----------------------------------------
qa = subprocess.run(f"cd {REPO} && {PY} scripts/phase2c_collection_qa.py --out {OUT}", shell=True, capture_output=True, text=True)
print(qa.stdout[-6000:]); print(qa.stderr[-2000:])
rep = json.load(open(f"{OUT}/qa_report.json"))
print("HARD FAILURES:", len(rep["hard_failures"]))
print("STATUS:", "COMPLETE" if rep["all_planned_complete"] else
      f"INCOMPLETE ({rep['complete_branch_points']}/{rep['planned_branch_points']}) -> run a second session (see top cell)")


In [ ]:
# ---- 8. package for download ---------------------------------------------------
!cd {WORK_OUT} && rm -f {os.path.basename(OUT)}.zip && zip -qr {os.path.basename(OUT)}.zip {os.path.basename(OUT)}
!ls -lh {WORK_OUT}/{os.path.basename(OUT)}.zip; du -sh {OUT}
